In [1]:
import sqlite3
import pandas as pd

# Connect to the database the script created
conn = sqlite3.connect("../data/nhs.db")

# 1. What tables do we have?
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)
print(tables)

                                       name
0                    april_2025_csv_revised
1                   august_2025_csv_revised
2                  february_2026_csv_dl8t54
3                     july_2025_csv_revised
4                     june_2025_csv_revised
5                      may_2025_csv_revised
6                    monthly_ae_august_2024
7                  monthly_ae_december_2024
8    monthly_ae_december_2025_revised_g9hy6
9          monthly_ae_february_2025_revised
10                  monthly_ae_january_2025
11    monthly_ae_january_2026_revised_3dker
12             monthly_ae_july_2024_revised
13                     monthly_ae_june_2024
14                    monthly_ae_march_2025
15     monthly_ae_march_2026_revised_flkg42
16            monthly_ae_nov_25_csv_revised
17         monthly_ae_november_2024_revised
18          monthly_ae_october_2024_revised
19  monthly_ae_october_2025_revised_fly02sk
20                monthly_ae_september_2024
21               september_2025_

In [2]:
# Get the list of table names from the database
table_names = tables["name"].tolist()

# Get the column names for every table
columns_by_table = {}
for t in table_names:
    cols = pd.read_sql(f"SELECT * FROM {t} LIMIT 0", conn).columns.tolist()
    columns_by_table[t] = cols

# Use the first table as the reference
reference_table = table_names[0]
reference_cols = set(columns_by_table[reference_table])

print(f"Reference table: {reference_table}")
print(f"Number of columns: {len(reference_cols)}\n")

# Compare every other table to the reference
for t in table_names:
    cols = set(columns_by_table[t])
    missing = reference_cols - cols
    extra = cols - reference_cols
    if missing or extra:
        print(f"DIFFERENT: {t}")
        print(f"  missing: {sorted(missing)}")
        print(f"  extra:   {sorted(extra)}")
    else:
        print(f"same:      {t}")

Reference table: april_2025_csv_revised
Number of columns: 22

same:      april_2025_csv_revised
same:      august_2025_csv_revised
same:      february_2026_csv_dl8t54
same:      july_2025_csv_revised
same:      june_2025_csv_revised
same:      may_2025_csv_revised
same:      monthly_ae_august_2024
same:      monthly_ae_december_2024
same:      monthly_ae_december_2025_revised_g9hy6
same:      monthly_ae_february_2025_revised
same:      monthly_ae_january_2025
same:      monthly_ae_january_2026_revised_3dker
same:      monthly_ae_july_2024_revised
same:      monthly_ae_june_2024
same:      monthly_ae_march_2025
same:      monthly_ae_march_2026_revised_flkg42
same:      monthly_ae_nov_25_csv_revised
same:      monthly_ae_november_2024_revised
same:      monthly_ae_october_2024_revised
same:      monthly_ae_october_2025_revised_fly02sk
DIFFERENT: monthly_ae_september_2024
  missing: []
  extra:   ['a', 'unnamed_22', 'unnamed_23', 'unnamed_24', 'unnamed_25', 'unnamed_26']
same:      septe

In [3]:
# Print the column names of the reference table
print(columns_by_table[reference_table])

# Show the first 5 rows
pd.read_sql(f"SELECT * FROM {reference_table} LIMIT 5", conn)

['period', 'org_code', 'parent_org', 'org_name', 'a_e_attendances_type_1', 'a_e_attendances_type_2', 'a_e_attendances_other_a_e_department', 'a_e_attendances_booked_appointments_type_1', 'a_e_attendances_booked_appointments_type_2', 'a_e_attendances_booked_appointments_other_department', 'attendances_over_4hrs_type_1', 'attendances_over_4hrs_type_2', 'attendances_over_4hrs_other_department', 'attendances_over_4hrs_booked_appointments_type_1', 'attendances_over_4hrs_booked_appointments_type_2', 'attendances_over_4hrs_booked_appointments_other_department', 'patients_who_have_waited_4_12_hs_from_dta_to_admission', 'patients_who_have_waited_12_hrs_from_dta_to_admission', 'emergency_admissions_via_a_e_type_1', 'emergency_admissions_via_a_e_type_2', 'emergency_admissions_via_a_e_other_a_e_department', 'other_emergency_admissions']


,period,org_code,parent_org,org_name,a_e_attendances_type_1,a_e_attendances_type_2,a_e_attendances_other_a_e_department,a_e_attendances_booked_appointments_type_1,a_e_attendances_booked_appointments_type_2,a_e_attendances_booked_appointments_other_department,...,attendances_over_4hrs_other_department,attendances_over_4hrs_booked_appointments_type_1,attendances_over_4hrs_booked_appointments_type_2,attendances_over_4hrs_booked_appointments_other_department,patients_who_have_waited_4_12_hs_from_dta_to_admission,patients_who_have_waited_12_hrs_from_dta_to_admission,emergency_admissions_via_a_e_type_1,emergency_admissions_via_a_e_type_2,emergency_admissions_via_a_e_other_a_e_department,other_emergency_admissions
0,MSitAE-APRIL-2025,NDJ,NHS ENGLAND SOUTH EAST,FIRST COMMUNITY HEALTH AND CARE CIC,0,0,2350,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,MSitAE-APRIL-2025,NTV0B,NHS ENGLAND SOUTH EAST,ASHFORD WALK-IN-CENTRE,0,0,2333,0,0,0,...,49,0,0,0,0,0,0,0,0,0
2,MSitAE-APRIL-2025,NTV0W,NHS ENGLAND SOUTH EAST,WOKING WALK IN CENTRE,0,0,2528,0,0,0,...,58,0,0,0,0,0,0,0,0,0
3,MSitAE-APRIL-2025,Y02615,NHS ENGLAND MIDLANDS,SOUTH BIRMINGHAM GP WALK IN CENTRE,0,0,6628,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,MSitAE-APRIL-2025,AQN04,NHS ENGLAND SOUTH EAST,PHL LYMINGTON UTC,0,0,3297,0,0,12,...,42,0,0,0,0,0,0,0,0,0


##  Combine the Monthly Tables

Step 1: Check the stray columns are empty

In [16]:
# Do the extra columns in September 2024 contain any data?
sep = pd.read_sql("SELECT * FROM monthly_ae_september_2024", conn)
extra_cols = ["a", "unnamed_22", "unnamed_23", "unnamed_24", "unnamed_25", "unnamed_26"]
print(sep[extra_cols].notna().sum())   # count of non-empty values per column

a             0
unnamed_22    0
unnamed_23    0
unnamed_24    0
unnamed_25    0
unnamed_26    0
dtype: int64


Step 2: Stack all 22 tables into one

In [17]:
reference_cols = columns_by_table["april_2025_csv_revised"]   # the 22 standard columns

frames = []
for t in table_names:
    df = pd.read_sql(f"SELECT * FROM {t}", conn)
    df = df[reference_cols]          # keep only the standard columns (drops the stray ones)
    df["source_table"] = t           # remember which file each row came from
    frames.append(df)

ae = pd.concat(frames, ignore_index=True)
print(ae.shape)

(4386, 23)


Step 3: Turn period into a proper date

In [18]:
# "MSitAE-APRIL-2025" -> month name "APRIL", year "2025"
parts = ae["period"].str.split("-", expand=True)
ae["month"] = pd.to_datetime(parts[1] + " " + parts[2], format="%B %Y", errors="coerce")

# Check: any rows we couldn't convert?
print("Unparsed dates:", ae["month"].isna().sum())

# Check: rows per month (should be roughly similar each month)
print(ae.groupby("month").size())

Unparsed dates: 21
month
2024-06-01    197
2024-07-01    198
2024-08-01    198
2024-09-01    197
2024-10-01    198
2024-11-01    198
2024-12-01    198
2025-01-01    198
2025-02-01    198
2025-03-01    198
2025-04-01    198
2025-05-01    200
2025-06-01    201
2025-07-01    201
2025-08-01    201
2025-09-01    201
2025-10-01    197
2025-11-01    197
2025-12-01    197
2026-01-01    197
2026-02-01    198
2026-03-01    199
dtype: int64


Step 4: Save it as one clean table

In [19]:
ae.to_sql("ae_monthly", conn, if_exists="replace", index=False)
pd.read_sql("SELECT COUNT(*) AS rows FROM ae_monthly", conn)

,rows
0,4386


Make the Table reproduceable from raw CSVs

In [21]:
from pathlib import Path
import sqlite3
import pandas as pd

DB = Path("../data/nhs.db")    # notebook is in notebooks/, so go up one level
conn = sqlite3.connect(DB)
# List the monthly tables created by load_data.py (skip any existing ae_monthly)
names = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' AND name != 'ae_monthly'", conn
)["name"].tolist()

# Use the first table's columns as the standard set
std_cols = pd.read_sql(f"SELECT * FROM {names[0]} LIMIT 0", conn).columns.tolist()

frames = []
for t in names:
    df = pd.read_sql(f"SELECT * FROM {t}", conn)[std_cols]  # drops stray columns
    df["source_table"] = t
    frames.append(df)

ae = pd.concat(frames, ignore_index=True)

# "MSitAE-APRIL-2025" -> real date
parts = ae["period"].str.split("-", expand=True)
ae["month"] = pd.to_datetime(parts[1] + " " + parts[2], format="%B %Y", errors="coerce")

ae.to_sql("ae_monthly", conn, if_exists="replace", index=False)
print(f"ae_monthly built: {len(ae)} rows, {ae['month'].nunique()} months")

ae_monthly built: 4386 rows, 22 months
